In [1]:
# -*- coding: utf-8 -*-
"""
Parts Discrepancy Investigator

Standalone Python version for Google Colab or a local Python environment.

How to use in Google Colab:
1. Upload this .py file to Colab, or keep it in Google Drive.
2. Run:
   !python parts_discrepancy_investigator_colab.py
3. The script will try to mount Google Drive automatically.
4. Open the Gradio URL shown by Colab.

Expected folder structure:
- One folder contains a *_list_*.md file.
- The same folder contains the six CSV files referenced by that list file.
- The output will be saved as investigation_queue.csv in that same folder.
"""

import sys
import subprocess
import importlib


def _ensure_package(package_name: str, import_name: str | None = None) -> None:
    """Install a package only when it is missing."""
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


_ensure_package("pandas")
_ensure_package("gradio")


# Mount Google Drive when running inside Google Colab.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped. This is normal outside Google Colab.")
    print(f"Mount message: {exc}")




import os
import re
import glob
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import gradio as gr


# -----------------------------
# Configuration
# -----------------------------

DEFAULT_FOLDER = "/content/drive/My Drive/Colab Notebooks/data/part06"

REQUIRED_COLUMNS = {
    "repair_orders": [
        "order_id", "shop_code", "shop_name", "vehicle_id", "opened_at",
        "service_line", "status", "service_advisor"
    ],
    "vehicle_releases": [
        "release_id", "order_id", "shop_code", "completed_at", "released_at",
        "released_by", "odometer", "release_status"
    ],
    "part_requirements": [
        "requirement_id", "order_id", "shop_code", "part_no",
        "required_qty", "requirement_status", "source"
    ],
    "parts_issues": [
        "issue_id", "issue_at", "shop_code", "order_id", "part_no",
        "issued_qty", "issued_by", "bin", "issue_status"
    ],
    "cycle_counts": [
        "count_id", "count_date", "shop_code", "part_no",
        "system_qty", "physical_qty", "counted_by", "count_reason"
    ],
    "part_master": [
        "part_no", "part_desc", "category", "unit_cost", "criticality", "uom"
    ],
}

DATE_COLUMNS = {
    "repair_orders": ["opened_at"],
    "vehicle_releases": ["completed_at", "released_at"],
    "part_requirements": [],
    "parts_issues": ["issue_at"],
    "cycle_counts": ["count_date"],
    "part_master": [],
}

ROLE_PATTERNS = {
    "repair_orders": "repair_orders",
    "vehicle_releases": "vehicle_releases",
    "part_requirements": "part_requirements",
    "parts_issues": "parts_issues",
    "cycle_counts": "cycle_counts",
    "part_master": "part_master",
}


# -----------------------------
# Small helper functions
# -----------------------------

def normalize_text_series(series: pd.Series) -> pd.Series:
    """Make key columns safer for matching."""
    return series.fillna("").astype(str).str.strip()


def scan_list_files(folder_path: str) -> List[str]:
    """Find all *_list_*.md files in the selected folder."""
    folder = Path(folder_path).expanduser()
    if not folder.exists():
        return []
    return sorted(str(p) for p in folder.glob("*_list_*.md"))


def parse_book_list(list_file_path: str) -> Dict[str, str]:
    """
    Read a markdown list file and identify the six CSV files inside it.

    The list file may contain absolute-looking paths such as /data/part06/file.csv.
    In Colab, the CSV files are expected to live in the same Drive folder as the list file,
    so this function falls back to using the basename inside the list file's folder.
    """
    list_path = Path(list_file_path)
    list_dir = list_path.parent

    text = list_path.read_text(encoding="utf-8")
    csv_tokens = re.findall(r"([^\s\]\)\"\'`]+\.csv)", text, flags=re.IGNORECASE)

    # Preserve order and remove duplicates.
    csv_tokens = list(dict.fromkeys(csv_tokens))

    role_to_path = {}
    for token in csv_tokens:
        raw_path = Path(token)

        # Prefer a path that already exists.
        if raw_path.exists():
            resolved = raw_path
        else:
            # Most reliable in this project: CSV files are in the same folder as the list file.
            same_folder = list_dir / raw_path.name
            if same_folder.exists():
                resolved = same_folder
            else:
                # Also try path relative to the list file folder.
                relative_to_list = list_dir / token
                resolved = relative_to_list

        filename_lower = resolved.name.lower()
        for role, pattern in ROLE_PATTERNS.items():
            if pattern in filename_lower:
                role_to_path[role] = str(resolved)

    missing_roles = sorted(set(REQUIRED_COLUMNS) - set(role_to_path))
    if missing_roles:
        found = "\n".join(csv_tokens) if csv_tokens else "(no csv paths found)"
        raise ValueError(
            "The selected list file does not identify all six required CSV files.\n"
            f"Missing roles: {missing_roles}\n\n"
            f"CSV paths found in the list file:\n{found}"
        )

    return role_to_path


def parse_date_columns(df: pd.DataFrame, role: str) -> Tuple[pd.DataFrame, List[dict]]:
    """Parse configured date columns and return a parsing report."""
    df = df.copy()
    results = []

    for col in DATE_COLUMNS.get(role, []):
        if col not in df.columns:
            results.append({
                "column": col,
                "raw_non_null": 0,
                "parsed_non_null": 0,
                "invalid_after_parse": "missing column",
            })
            continue

        raw_non_null = int(df[col].notna().sum())
        parsed = pd.to_datetime(df[col], errors="coerce")
        invalid_after_parse = int(parsed.isna().sum() - df[col].isna().sum())

        df[col] = parsed
        results.append({
            "column": col,
            "raw_non_null": raw_non_null,
            "parsed_non_null": int(parsed.notna().sum()),
            "invalid_after_parse": invalid_after_parse,
        })

    return df, results


def validate_columns(df: pd.DataFrame, role: str) -> List[str]:
    """Return a list of missing required columns."""
    required = REQUIRED_COLUMNS[role]
    return [col for col in required if col not in df.columns]


def load_and_inspect(list_file_path: str) -> Tuple[Dict[str, pd.DataFrame], str]:
    """
    Load all CSVs from the selected list file.
    Return the dataframes and a human-readable structure report.
    """
    role_to_path = parse_book_list(list_file_path)

    loaded = {}
    report_lines = []
    report_lines.append(f"# Stage 1 — Load files and inspect structure")
    report_lines.append(f"Selected list file: `{list_file_path}`")
    report_lines.append("")

    for role in REQUIRED_COLUMNS.keys():
        csv_path = role_to_path[role]

        if not Path(csv_path).exists():
            raise FileNotFoundError(
                f"CSV file for {role} was listed but not found:\n{csv_path}\n\n"
                "Check that the CSV is in the same Google Drive folder as the selected list file."
            )

        df = pd.read_csv(csv_path)
        missing_cols = validate_columns(df, role)
        df, date_report = parse_date_columns(df, role)
        loaded[role] = df

        report_lines.append(f"## {role}")
        report_lines.append(f"- Path: `{csv_path}`")
        report_lines.append(f"- Rows: {len(df):,}")
        report_lines.append(f"- Columns: `{', '.join(df.columns)}`")

        if missing_cols:
            report_lines.append(f"- Missing required columns: **{', '.join(missing_cols)}**")
        else:
            report_lines.append("- Required column check: OK")

        dtype_text = ", ".join([f"{c}: {t}" for c, t in df.dtypes.astype(str).items()])
        report_lines.append(f"- Data types: `{dtype_text}`")

        missing_text = ", ".join([f"{c}: {int(v)}" for c, v in df.isna().sum().items()])
        report_lines.append(f"- Missing values: `{missing_text}`")

        if date_report:
            report_lines.append("- Date parsing:")
            for item in date_report:
                report_lines.append(
                    f"  - `{item['column']}`: raw non-null={item['raw_non_null']}, "
                    f"parsed non-null={item['parsed_non_null']}, "
                    f"invalid after parse={item['invalid_after_parse']}"
                )
        else:
            report_lines.append("- Date parsing: no configured date columns")

        report_lines.append("")

    return loaded, "\n".join(report_lines)


def fill_missing_issue_status(parts_issues: pd.DataFrame, repair_orders: pd.DataFrame) -> pd.DataFrame:
    """
    If issue_status is empty, classify the record enough for investigation.
    Existing non-empty statuses are preserved.
    """
    issues = parts_issues.copy()

    if "issue_status" not in issues.columns:
        issues["issue_status"] = ""

    valid_order_ids = set(normalize_text_series(repair_orders["order_id"]))
    order_id = normalize_text_series(issues["order_id"])
    status = normalize_text_series(issues["issue_status"])
    empty_status = status.eq("")

    issues.loc[empty_status & order_id.eq(""), "issue_status"] = "orphan_blank_order"
    issues.loc[
        empty_status & order_id.ne("") & ~order_id.isin(valid_order_ids),
        "issue_status"
    ] = "orphan_nonexistent_order"
    issues.loc[
        empty_status & order_id.ne("") & order_id.isin(valid_order_ids),
        "issue_status"
    ] = "posted"

    return issues


def build_investigation_queue(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Apply the detection rules, scoring rules, and note generation.
    The result has one row per shop_code + part_no combination.
    """
    repair_orders = data["repair_orders"].copy()
    vehicle_releases = data["vehicle_releases"].copy()
    part_requirements = data["part_requirements"].copy()
    parts_issues = fill_missing_issue_status(data["parts_issues"].copy(), repair_orders)
    cycle_counts = data["cycle_counts"].copy()
    part_master = data["part_master"].copy()

    # Normalize key columns.
    for df in [
        repair_orders,
        vehicle_releases,
        part_requirements,
        parts_issues,
        cycle_counts,
        part_master,
    ]:
        for col in ["order_id", "shop_code", "part_no", "requirement_status", "release_status", "criticality"]:
            if col in df.columns:
                df[col] = normalize_text_series(df[col])

    # Numeric safety.
    part_requirements["required_qty"] = pd.to_numeric(
        part_requirements["required_qty"], errors="coerce"
    ).fillna(0)
    parts_issues["issued_qty"] = pd.to_numeric(
        parts_issues["issued_qty"], errors="coerce"
    ).fillna(0)
    cycle_counts["system_qty"] = pd.to_numeric(
        cycle_counts["system_qty"], errors="coerce"
    )
    cycle_counts["physical_qty"] = pd.to_numeric(
        cycle_counts["physical_qty"], errors="coerce"
    )
    part_master["unit_cost"] = pd.to_numeric(
        part_master["unit_cost"], errors="coerce"
    ).fillna(0)

    # Stage 2: released vehicles are completed work.
    released = vehicle_releases[
        vehicle_releases["release_status"].str.lower().eq("released")
    ][["order_id", "shop_code", "released_at"]].drop_duplicates()

    completed_requirements = part_requirements[
        part_requirements["requirement_status"].str.lower().eq("completed")
        & (part_requirements["required_qty"] > 0)
    ].copy()

    released_requirements = completed_requirements.merge(
        released,
        on=["order_id", "shop_code"],
        how="inner",
    )

    # If the same order-part appears more than once, combine it before matching issue records.
    req_order_part = (
        released_requirements
        .groupby(["shop_code", "order_id", "part_no"], as_index=False)
        .agg(
            required_qty=("required_qty", "sum"),
            released_at=("released_at", "max"),
        )
    )

    # Linked issue records by same shop_code + order_id + part_no.
    issue_by_order_part = (
        parts_issues
        .groupby(["shop_code", "order_id", "part_no"], dropna=False, as_index=False)
        .agg(
            linked_issued_qty=("issued_qty", "sum"),
            linked_issue_count=("issue_id", "count"),
            latest_issue_at=("issue_at", "max"),
        )
    )

    requirement_detail = req_order_part.merge(
        issue_by_order_part,
        on=["shop_code", "order_id", "part_no"],
        how="left",
    )

    requirement_detail["linked_issue_count"] = (
        requirement_detail["linked_issue_count"].fillna(0).astype(int)
    )
    requirement_detail["linked_issued_qty"] = (
        requirement_detail["linked_issued_qty"].fillna(0)
    )

    requirement_detail["missing_link_flag"] = requirement_detail["linked_issue_count"].eq(0)
    requirement_detail["under_issued_flag"] = (
        requirement_detail["linked_issue_count"].gt(0)
        & (requirement_detail["linked_issued_qty"] < requirement_detail["required_qty"])
    )
    requirement_detail["late_issue_flag"] = (
        requirement_detail["linked_issue_count"].gt(0)
        & (
            requirement_detail["latest_issue_at"]
            > (requirement_detail["released_at"] + pd.Timedelta(minutes=30))
        )
    )

    req_agg = (
        requirement_detail
        .groupby(["shop_code", "part_no"], as_index=False)
        .agg(
            related_order_count=("order_id", "nunique"),
            expected_qty=("required_qty", "sum"),
            linked_issued_qty=("linked_issued_qty", "sum"),
            missing_link_count=("missing_link_flag", "sum"),
            under_issued_count=("under_issued_flag", "sum"),
            late_issue_count=("late_issue_flag", "sum"),
            order_ids=("order_id", lambda x: ", ".join(sorted(set(x)))),
            first_signal_date=("released_at", "min"),
        )
    )

    # Orphan issue: empty order_id or order_id not found in repair_orders.
    valid_order_ids = set(repair_orders["order_id"])
    issue_order_id = normalize_text_series(parts_issues["order_id"])

    orphan_issues = parts_issues[
        issue_order_id.eq("") | ~issue_order_id.isin(valid_order_ids)
    ].copy()

    if len(orphan_issues) > 0:
        orphan_agg = (
            orphan_issues
            .groupby(["shop_code", "part_no"], as_index=False)
            .agg(
                orphan_issue_qty=("issued_qty", "sum"),
                orphan_issue_count=("issue_id", "count"),
                orphan_signal_date=("issue_at", "min"),
            )
        )
    else:
        orphan_agg = pd.DataFrame(
            columns=[
                "shop_code", "part_no", "orphan_issue_qty",
                "orphan_issue_count", "orphan_signal_date"
            ]
        )

    # Latest cycle count per shop-part.
    if len(cycle_counts) > 0:
        latest_counts = (
            cycle_counts
            .sort_values(["shop_code", "part_no", "count_date"])
            .groupby(["shop_code", "part_no"], as_index=False)
            .tail(1)
            .copy()
        )
        latest_counts["count_gap"] = latest_counts["system_qty"] - latest_counts["physical_qty"]
        count_agg = latest_counts[
            ["shop_code", "part_no", "system_qty", "physical_qty", "count_gap", "count_date"]
        ].copy()
    else:
        count_agg = pd.DataFrame(
            columns=["shop_code", "part_no", "system_qty", "physical_qty", "count_gap", "count_date"]
        )

    # One row per observed shop_code + part_no.
    universe = pd.concat(
        [
            req_order_part[["shop_code", "part_no"]],
            parts_issues[["shop_code", "part_no"]],
            cycle_counts[["shop_code", "part_no"]],
        ],
        ignore_index=True,
    ).drop_duplicates()

    out = universe.merge(req_agg, on=["shop_code", "part_no"], how="left")
    out = out.merge(orphan_agg, on=["shop_code", "part_no"], how="left")
    out = out.merge(count_agg, on=["shop_code", "part_no"], how="left")
    out = out.merge(
        part_master[["part_no", "part_desc", "criticality", "unit_cost"]],
        on="part_no",
        how="left",
    )

    # Defaults.
    zero_cols = [
        "related_order_count", "expected_qty", "linked_issued_qty",
        "missing_link_count", "under_issued_count", "late_issue_count",
        "orphan_issue_qty", "orphan_issue_count",
        "physical_qty", "system_qty", "count_gap", "unit_cost",
    ]
    for col in zero_cols:
        if col in out.columns:
            out[col] = out[col].fillna(0)

    out["order_ids"] = out["order_ids"].fillna("")
    out["criticality"] = out["criticality"].fillna("")
    out["part_desc"] = out["part_desc"].fillna("")

    # Recurring pattern: two or more operational signals for the same shop-part
    # inside the latest observed 30-day window. Cycle count confirmation alone
    # does not create the recurring flag.
    date_series = []
    for col in ["first_signal_date", "orphan_signal_date", "count_date"]:
        if col in out.columns:
            date_series.append(out[col].dropna())

    if date_series:
        all_dates = pd.concat(date_series, ignore_index=True)
        anchor_date = all_dates.max() if len(all_dates) else pd.Timestamp.now()
    else:
        anchor_date = pd.Timestamp.now()

    window_start = anchor_date - pd.Timedelta(days=30)

    recurring_flags = []
    for _, row in out.iterrows():
        dates = []

        if (
            row.get("missing_link_count", 0) > 0
            or row.get("under_issued_count", 0) > 0
            or row.get("late_issue_count", 0) > 0
        ):
            d = row.get("first_signal_date", pd.NaT)
            if pd.notna(d):
                dates.append(d)

        if row.get("orphan_issue_count", 0) > 0:
            d = row.get("orphan_signal_date", pd.NaT)
            if pd.notna(d):
                dates.append(d)

        dates = [d for d in dates if pd.notna(d) and window_start <= d <= anchor_date]
        recurring_flags.append(len(dates) >= 2)

    out["recurring_pattern_flag"] = recurring_flags

    # Stage 3: scoring.
    out["discrepancy_score"] = 0

    out.loc[out["missing_link_count"] > 0, "discrepancy_score"] += 40
    out.loc[out["under_issued_count"] > 0, "discrepancy_score"] += 30
    out.loc[out["orphan_issue_count"] > 0, "discrepancy_score"] += 20
    out.loc[out["late_issue_count"] > 0, "discrepancy_score"] += 15
    out.loc[out["count_gap"] > 0, "discrepancy_score"] += 15
    out.loc[out["recurring_pattern_flag"], "discrepancy_score"] += 10

    # Criticality and cost are used as amplifiers only when an operational
    # pre-count signal exists. This prevents a cycle count shortage alone from
    # dominating the queue.
    operational_signal = (
        (out["missing_link_count"] > 0)
        | (out["under_issued_count"] > 0)
        | (out["orphan_issue_count"] > 0)
        | (out["late_issue_count"] > 0)
    )

    out.loc[
        operational_signal & out["criticality"].str.upper().eq("HIGH"),
        "discrepancy_score",
    ] += 10
    out.loc[
        operational_signal & (out["unit_cost"] >= 50),
        "discrepancy_score",
    ] += 5

    out["discrepancy_score"] = out["discrepancy_score"].clip(lower=0).astype(int)

    def severity_from_score(score: int) -> str:
        if score >= 70:
            return "URGENT"
        if score >= 40:
            return "REVIEW_SOON"
        return "MONITOR"

    out["severity"] = out["discrepancy_score"].apply(severity_from_score)

    def cause_category(row: pd.Series) -> str:
        if row["missing_link_count"] > 0:
            return "MISSING_LINKED_ISSUE"
        if row["under_issued_count"] > 0:
            return "UNDER_ISSUED"
        if row["orphan_issue_count"] > 0:
            return "ORPHAN_ISSUE"
        if row["late_issue_count"] > 0:
            return "LATE_ISSUE"
        if row["recurring_pattern_flag"]:
            return "REPEATED_PATTERN"
        if row["count_gap"] > 0:
            return "CYCLE_COUNT_CONFIRMATION"
        return "NO_MATERIAL_SIGNAL"

    out["cause_category"] = out.apply(cause_category, axis=1)

    # Stage 4: coordinator notes.
    def make_notes(row: pd.Series) -> pd.Series:
        signals = []
        actions = []

        if row["missing_link_count"] > 0:
            signals.append(
                f"{int(row['missing_link_count'])} released order-part line has no linked issue record"
            )
            actions.append(
                "check whether the part was issued under another order, a manual ticket, or an unposted issue"
            )

        if row["under_issued_count"] > 0:
            signals.append(
                f"{int(row['under_issued_count'])} linked issue line is below required quantity"
            )
            actions.append(
                "compare the repair file, estimate card, and issue ticket quantity"
            )

        if row["orphan_issue_count"] > 0:
            signals.append(
                f"{int(row['orphan_issue_count'])} issue record cannot be tied to a valid repair order"
            )
            actions.append(
                "confirm the intended order number or whether this was a shop-floor adjustment"
            )

        if row["late_issue_count"] > 0:
            signals.append(
                f"{int(row['late_issue_count'])} issue record was posted more than 30 minutes after release"
            )
            actions.append(
                "confirm whether the issue was entered after the vehicle left"
            )

        if row["count_gap"] > 0:
            signals.append(
                f"cycle count shows physical quantity lower than system by {int(row['count_gap'])}"
            )
            actions.append(
                "use the count result as confirmation evidence after checking earlier transaction records"
            )

        if row["recurring_pattern_flag"]:
            signals.append(
                "more than one operational discrepancy signal appears in the recent window"
            )
            actions.append(
                "review repeated handling pattern for this shop-part combination"
            )

        if not signals:
            return pd.Series({
                "likely_cause": (
                    "No material discrepancy signal found from released-order requirements, "
                    "issue records, or the latest cycle count."
                ),
                "reviewer_action": "No immediate action. Keep this item on normal monitoring.",
            })

        return pd.Series({
            "likely_cause": (
                "Potential investigation item: "
                + "; ".join(signals)
                + ". This is not a final cause determination."
            ),
            "reviewer_action": (
                "Next confirmation: "
                + "; ".join(dict.fromkeys(actions))
                + "."
            ),
        })

    out[["likely_cause", "reviewer_action"]] = out.apply(make_notes, axis=1)

    final_columns = [
        "shop_code",
        "part_no",
        "part_desc",
        "related_order_count",
        "expected_qty",
        "linked_issued_qty",
        "orphan_issue_qty",
        "orphan_issue_count",
        "late_issue_count",
        "physical_qty",
        "system_qty",
        "count_gap",
        "criticality",
        "unit_cost",
        "cause_category",
        "discrepancy_score",
        "severity",
        "order_ids",
        "likely_cause",
        "reviewer_action",
    ]

    out = (
        out[final_columns]
        .sort_values(
            ["discrepancy_score", "shop_code", "part_no"],
            ascending=[False, True, True],
        )
        .reset_index(drop=True)
    )

    return out


def run_investigation(list_file_path: str):
    """
    Main function used by Gradio.
    Returns:
    - structure report
    - result summary
    - dataframe for preview
    - CSV file path for download
    """
    if not list_file_path:
        raise gr.Error("Select a *_list_*.md file first.")

    data, structure_report = load_and_inspect(list_file_path)
    result = build_investigation_queue(data)

    output_path = str(Path(list_file_path).parent / "investigation_queue.csv")
    result.to_csv(output_path, index=False, encoding="utf-8-sig")

    top_preview = result.head(10)[
        ["shop_code", "part_no", "cause_category", "discrepancy_score", "severity", "reviewer_action"]
    ]

    summary_lines = [
        "# Stage 2–4 — Detection, scoring, and saved result",
        f"- Output file: `{output_path}`",
        f"- Result rows: {len(result):,}",
        f"- Minimum score: {int(result['discrepancy_score'].min()) if len(result) else 0}",
        f"- Maximum score: {int(result['discrepancy_score'].max()) if len(result) else 0}",
        "",
        "## Top investigation items",
        top_preview.to_markdown(index=False),
    ]

    return structure_report, "\n".join(summary_lines), result, output_path


# -----------------------------
# Gradio app
# -----------------------------

def refresh_dropdown(folder_path: str):
    files = scan_list_files(folder_path)
    if not files:
        return gr.update(choices=[], value=None), "No *_list_*.md files found in this folder."
    return gr.update(choices=files, value=files[0]), f"Found {len(files)} list file(s)."


with gr.Blocks(title="Parts Discrepancy Investigator") as demo:
    gr.Markdown(
        """
        # Parts Discrepancy Investigator

        1. Enter the Google Drive folder that contains the `*_list_*.md` file and the six CSV files.
        2. Click **Scan folder**.
        3. Select a list file.
        4. Click **Run investigation**.

        The output file will be saved as `investigation_queue.csv` in the same folder.
        """
    )

    with gr.Row():
        folder_box = gr.Textbox(
            label="Google Drive folder",
            value=DEFAULT_FOLDER,
            lines=1,
        )
        scan_button = gr.Button("Scan folder")

    scan_status = gr.Markdown()
    list_dropdown = gr.Dropdown(
        label="Select *_list_*.md file",
        choices=[],
        value=None,
        interactive=True,
    )

    run_button = gr.Button("Run investigation", variant="primary")

    with gr.Tab("Stage 1 — Structure inspection"):
        structure_report = gr.Markdown()

    with gr.Tab("Stage 2–4 — Result summary"):
        result_summary = gr.Markdown()

    with gr.Tab("Investigation queue preview"):
        result_table = gr.Dataframe(
            label="investigation_queue.csv preview",
            wrap=True,
            interactive=False,
        )

    output_file = gr.File(label="Download investigation_queue.csv")

    scan_button.click(
        refresh_dropdown,
        inputs=[folder_box],
        outputs=[list_dropdown, scan_status],
    )

    run_button.click(
        run_investigation,
        inputs=[list_dropdown],
        outputs=[structure_report, result_summary, result_table, output_file],
    )


demo.launch(share=False, debug=True)



Mounted at /content/drive
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
